In [ ]:
# 1. Install core libraries (run once per session)
#!pip install --quiet pandas numpy scikit-learn transformers torch matplotlib


In [ ]:
# 2. Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import email
from email import policy

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Transformers (DistilBERT + summarisation)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM



In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Global configuration

DATA_PATH   = "/content/drive/MyDrive/MSc_Email_Project/emails.csv"
TEST_SIZE   = 0.2    # 80% train, 20% test
RANDOM_STATE = 42
N_ROWS      = 50000


In [ ]:
import random
import torch

def set_seed(seed: int = RANDOM_STATE):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def print_section(title: str):
    """Nice separator for notebook output."""
    print("\n")
    print(f"{title}")
    print("\n" )

set_seed()
#print_section("CONFIG & HELPERS INITIALISED")


In [ ]:
print_section("LOAD RAW ENRON CSV")

# Load raw data: expects columns ['file', 'message']
df_raw = pd.read_csv(DATA_PATH, nrows=N_ROWS)

print(f"Dataset loaded successfully. Rows: {len(df_raw)}, Columns: {df_raw.shape[1]}")
df_raw.head()




LOAD RAW ENRON CSV


Dataset loaded successfully. Rows: 50000, Columns: 2


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [ ]:
# Quick overview of the raw structure
print("Columns:", df_raw.columns.tolist())
print("\nSample file paths:")
print(df_raw['file'].head(10))

print("\nSample raw message (truncated):")
print(df_raw['message'].iloc[1][:1000])

Columns: ['file', 'message']

Sample file paths:
0       allen-p/_sent_mail/1.
1      allen-p/_sent_mail/10.
2     allen-p/_sent_mail/100.
3    allen-p/_sent_mail/1000.
4    allen-p/_sent_mail/1001.
5    allen-p/_sent_mail/1002.
6    allen-p/_sent_mail/1003.
7    allen-p/_sent_mail/1004.
8     allen-p/_sent_mail/101.
9     allen-p/_sent_mail/102.
Name: file, dtype: object

Sample raw message (truncated):
Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Traveling to have a business meeting takes the fun out of the trip.  Especially if you have to prepare

In [ ]:
print_section("PARSE EMAILS → HEADERS + BODY")

def parse_single_email(raw_text: str) -> pd.Series:
    """
    Parse one raw email string into headers + plain-text body.
    If parsing fails, fall back to using the raw text as body.
    """
    try:
        msg = email.message_from_string(raw_text, policy=policy.default)

        message_id = msg.get('Message-ID', '')
        date       = msg.get('Date', '')
        from_      = msg.get('From', '')
        to         = msg.get('To', '')
        subject    = msg.get('Subject', '')

        # Extract plain text body
        body = ""
        if msg.is_multipart():
            parts = []
            for part in msg.walk():
                if part.get_content_type() == "text/plain" and not part.get_content_disposition():
                    payload = part.get_payload(decode=True)
                    if isinstance(payload, bytes):
                        parts.append(payload.decode(errors="ignore"))
                    elif isinstance(payload, str):
                        parts.append(payload)
            body = "\n".join(parts)
        else:
            payload = msg.get_payload(decode=True)
            if isinstance(payload, bytes):
                body = payload.decode(errors="ignore")
            elif isinstance(payload, str):
                body = payload
            else:
                body = ""
    except Exception:
        # Fallback if anything goes wrong
        message_id = ""
        date = ""
        from_ = ""
        to = ""
        subject = ""
        body = raw_text

    return pd.Series({
        "message_id": message_id,
        "date": date,
        "from": from_,
        "to": to,
        "subject": subject,
        "body": body
    })

# Apply parser to every message
parsed = df_raw["message"].apply(parse_single_email)

# Combine with original file column
df = pd.concat([df_raw.reset_index(drop=True), parsed], axis=1)

print("Parsed columns now:", df.columns.tolist())
df[['file', 'message_id', 'date', 'from', 'to', 'subject']].head()





PARSE EMAILS → HEADERS + BODY


Parsed columns now: ['file', 'message', 'message_id', 'date', 'from', 'to', 'subject', 'body']


,file,message_id,date,from,to,subject
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700",phillip.allen@enron.com,tim.belden@enron.com,
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 04 May 2001 13:51:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,Re:
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test
3,allen-p/_sent_mail/1000.,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700",phillip.allen@enron.com,randall.gay@enron.com,
4,allen-p/_sent_mail/1001.,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello


In [ ]:
# or, without file + message_id:
df[['date', 'from', 'to', 'subject', 'body']].head()

,date,from,to,subject,body
0,"Mon, 14 May 2001 16:39:00 -0700",phillip.allen@enron.com,tim.belden@enron.com,,Here is our forecast\n\n
1,"Fri, 04 May 2001 13:51:00 -0700",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,"Wed, 18 Oct 2000 03:00:00 -0700",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,"Mon, 23 Oct 2000 06:13:00 -0700",phillip.allen@enron.com,randall.gay@enron.com,,"Randy,\n\n Can you send me a schedule of the s..."
4,"Thu, 31 Aug 2000 05:07:00 -0700",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [ ]:
print_section("CREATE LABEL (folder_label)")

def get_folder_from_path(path: str) -> str:
    """
    Derive a folder label from the Enron file path.
    Examples:
      'allen-p/_sent_mail/1.'     -> 'sent'
      'allen-p/inbox/5.'         -> 'inbox'
      'allen-p/deleted_items/3.' -> 'deleted'
    Everything else -> 'other'
    """
    p = str(path).lower()
    if "sent_mail" in p or "/sent" in p:
        return "sent"
    if "inbox" in p:
        return "inbox"
    if "deleted_items" in p or "deleted" in p:
        return "deleted"
    if "drafts" in p:
        return "drafts"
    return "other"

# Apply the function to create the label column
df["folder_label"] = df["file"].apply(get_folder_from_path)

print("Folder label counts (before filtering):")
print(df["folder_label"].value_counts())




CREATE LABEL (folder_label)


Folder label counts (before filtering):
folder_label
other      24188
sent       13342
inbox       7529
deleted     4938
drafts         3
Name: count, dtype: int64


In [ ]:
df[df["folder_label"] == "other"][["file", "subject", "body"]].sample(10, random_state=42)


,file,subject,body
32380,benson-r/all_documents/25.,EPMI-Southeast,Please remind everyone that works for you that...
45796,campbell-l/personnal/36.,Program Changes,"\nAs you know, this is an unprecedented time i..."
12892,bass-e/all_documents/55.,Fwd: Lifesavers,--\nAmir Cyrus Ahanchian\nacahanchian@zdnetone...
28510,beck-s/eol/54.,Re: EOL Revenues,Call with any questions or comments. Georgann...
22740,beck-s/all_documents/2034.,Re: Personnel for EBS,I am not close enough to know Bob's exact role...
27218,beck-s/discussion_threads/3761.,Re: OU Scholarships,This looks great. Do you need me for either o...
24875,beck-s/calendar/100.,281.367.1100 - Center for Houston's Future,"\n\nbreakfast, work, lunch, work, break at 3pm"
24935,beck-s/calendar/41.,Cassandra Shultz - Update (RC x30402),\n\nrequested by Cassandra - personal mtg.
34316,blair-l/meetings___nng_customer_mtg/17.,Winter Ops. Meeting,Below are the Customer Service Representatives...
15018,bass-e/discussion_threads/843.,New Gore Campaign Sign,\n\n\nBrian T. Hoskins\nEnron Broadband Servic...


In [ ]:
print_section("PREPROCESS TEXT (subject + body)")

def preprocess_text_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Clean subject/body and create a combined text field.
    This cleaned dataframe will be reused for BOTH
    classification and summarisation.
    """
    df_out = df_in.copy()

    # Basic cleaning: handle missing, convert to string, strip spaces
    df_out["subject"] = (
        df_out["subject"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df_out["body"] = (
        df_out["body"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Combined text feature for models
    df_out["text"] = (df_out["subject"] + " " + df_out["body"]).str.strip()

    # Simple length features (for EDA & summarisation filters later)
    df_out["subject_len"] = df_out["subject"].str.len()
    df_out["body_len"]    = df_out["body"].str.len()
    df_out["text_len"]    = df_out["text"].str.len()

    return df_out

#  Master cleaned dataset – KEEP ALL FOLDERS HERE
df_clean = preprocess_text_columns(df)

print("df_clean shape:", df_clean.shape)
df_clean[["folder_label", "subject", "body"]].tail()




PREPROCESS TEXT (subject + body)


df_clean shape: (50000, 13)


,folder_label,subject,body
49995,sent,FW: Pam Benson,"please file ""Pam Benson"". Thanks. MHC\n\n --..."
49996,sent,FW: ECI,please file in Diomedes' file. Thanks. MHC\n...
49997,sent,RE: In re Event Resources,fine with me. mhc\n\n-----Original Message---...
49998,sent,RE: List of People,You need more than the 840 for this -- you als...
49999,sent,RE: Question from Amy Fitzpatrick,No risk to non-compete. The payment obligatio...


In [ ]:
print_section(" BUILD CLASSIFICATION DATASET + TRAIN/TEST SPLIT")

# 1. Classification view: only main folders
main_folders = ["sent", "inbox", "deleted"]
df_clf = df_clean[df_clean["folder_label"].isin(main_folders)].copy()

print("df_clf shape BEFORE length filtering:", df_clf.shape)
print("\nFolder label counts (classification dataset):")
print(df_clf["folder_label"].value_counts())

# 1a. Remove extremely short emails (optional design choice)
MIN_TEXT_LEN = 5     # remove text_len < 5
mask_short = df_clf["text_len"] < MIN_TEXT_LEN
print(f"\nVery short emails (text_len < {MIN_TEXT_LEN}):", mask_short.sum())

df_clf = df_clf[~mask_short].copy()
print("df_clf shape AFTER length filtering:", df_clf.shape)

# 2. TEXT features and labels
X_text = df_clf["text"]
y      = df_clf["folder_label"]

print_section("CHECK TEXT LENGTH OUTLIERS")

# Basic stats for text length in the classification dataset
print("Text length stats (classification dataset):")
print(df_clf["text_len"].describe())

# Look at extreme quantiles
quantiles = df_clf["text_len"].quantile([0.01, 0.05, 0.95, 0.99])

# Define simple rules for 'very short' and 'very long' texts
VERY_SHORT_THRESHOLD = 5      # chars
VERY_LONG_THRESHOLD  = quantiles.loc[0.99]  # top 1% as 'very long'

very_short_mask = df_clf["text_len"] <= VERY_SHORT_THRESHOLD
very_long_mask  = df_clf["text_len"] >= VERY_LONG_THRESHOLD

n_very_short = very_short_mask.sum()
n_very_long  = very_long_mask.sum()

print(f"\nVery short emails (text_len <= {VERY_SHORT_THRESHOLD}): {n_very_short}")
print(f"Very long emails (text_len >= {VERY_LONG_THRESHOLD:.0f}): {n_very_long}")

print("\nSample VERY SHORT emails:")
display(df_clf.loc[very_short_mask, ["folder_label", "subject", "body", "text_len"]].head(5))

print("\nSample VERY LONG emails:")
display(df_clf.loc[very_long_mask, ["folder_label", "subject", "body_len", "text_len"]].head(5))

# 3. Single train/test split – reused for ALL models
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTrain size:", X_train_text.shape[0])
print("Test size :", X_test_text.shape[0])

train_idx = X_train_text.index
test_idx  = X_test_text.index




 BUILD CLASSIFICATION DATASET + TRAIN/TEST SPLIT


df_clf shape BEFORE length filtering: (25809, 13)

Folder label counts (classification dataset):
folder_label
sent       13342
inbox       7529
deleted     4938
Name: count, dtype: int64

Very short emails (text_len < 5): 3
df_clf shape AFTER length filtering: (25806, 13)


CHECK TEXT LENGTH OUTLIERS


Text length stats (classification dataset):
count     25806.000000
mean       1902.613772
std        5582.572542
min           5.000000
25%         327.000000
50%         801.000000
75%        1762.000000
max      272055.000000
Name: text_len, dtype: float64

Very short emails (text_len <= 5): 5
Very long emails (text_len >= 19118): 259

Sample VERY SHORT emails:


,folder_label,subject,body,text_len
10946,sent,,well?,5
17215,sent,,well?,5
40031,sent,,PRICK,5
44259,inbox,hi,hi,5
46250,sent,,PRICK,5



Sample VERY LONG emails:


,folder_label,subject,body_len,text_len
1234,deleted,NT Earnings Information,39617,39641
1268,deleted,AXP Earnings Information,24477,24502
3602,sent,Enron Mentions,22597,22612
3618,sent,Enron Mentions - 03-04-01,31682,31708
4995,deleted,Enron Mentions,20528,20543



Train size: 20644
Test size : 5162


In [ ]:
print_section("PART 8 NAIVE BAYES")

# Define the Naive Bayes pipeline
nb_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

# Fit on the shared train split
nb_model.fit(X_train_text, y_train)

print("Naive Bayes model trained on", len(X_train_text), "emails.")




PART 8 NAIVE BAYES


Naive Bayes model trained on 20644 emails.


In [ ]:
print_section("NAIVE BAYES (evaluation)")

# Predictions
y_train_pred_nb = nb_model.predict(X_train_text)
y_test_pred_nb  = nb_model.predict(X_test_text)

# Metrics
nb_train_acc = accuracy_score(y_train, y_train_pred_nb)
nb_test_acc  = accuracy_score(y_test, y_test_pred_nb)
nb_train_f1  = f1_score(y_train, y_train_pred_nb, average="macro")
nb_test_f1   = f1_score(y_test, y_test_pred_nb, average="macro")

print("Naive Bayes – Train Acc:", f"{nb_train_acc:.3f}", "Test Acc:", f"{nb_test_acc:.3f}")
print("Naive Bayes – Train F1 :", f"{nb_train_f1:.3f}",  "Test F1 :", f"{nb_test_f1:.3f}")

print("\nNaive Bayes – Classification report :\n")
print(classification_report(y_test, y_test_pred_nb, digits=3, zero_division=0))




NAIVE BAYES (evaluation)


Naive Bayes – Train Acc: 0.680 Test Acc: 0.660
Naive Bayes – Train F1 : 0.537 Test F1 : 0.505

Naive Bayes – Classification report :

              precision    recall  f1-score   support

     deleted      0.799     0.120     0.209       988
       inbox      0.694     0.419     0.523      1506
        sent      0.647     0.995     0.784      2668

    accuracy                          0.660      5162
   macro avg      0.713     0.512     0.505      5162
weighted avg      0.690     0.660     0.598      5162



In [ ]:
print_section("LINEAR SVM")

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC())
])

svm_model.fit(X_train_text, y_train)

print("Linear SVM model trained on", len(X_train_text), "emails.")




LINEAR SVM


Linear SVM model trained on 20644 emails.


In [ ]:
print_section("LINEAR SVM (evaluation)")

y_train_pred_svm = svm_model.predict(X_train_text)
y_test_pred_svm  = svm_model.predict(X_test_text)

svm_train_acc = accuracy_score(y_train, y_train_pred_svm)
svm_test_acc  = accuracy_score(y_test, y_test_pred_svm)
svm_train_f1  = f1_score(y_train, y_train_pred_svm, average="macro")
svm_test_f1   = f1_score(y_test, y_test_pred_svm, average="macro")

print("Linear SVM – Train Acc:", f"{svm_train_acc:.3f}", "Test Acc:", f"{svm_test_acc:.3f}")
print("Linear SVM – Train F1 :", f"{svm_train_f1:.3f}",  "Test F1 :", f"{svm_test_f1:.3f}")

print("\nLinear SVM – Classification report (test set):\n")
print(classification_report(y_test, y_test_pred_svm, digits=3, zero_division=0))




LINEAR SVM (evaluation)


Linear SVM – Train Acc: 0.940 Test Acc: 0.793
Linear SVM – Train F1 : 0.925 Test F1 : 0.746

Linear SVM – Classification report (test set):

              precision    recall  f1-score   support

     deleted      0.657     0.602     0.628       988
       inbox      0.739     0.703     0.720      1506
        sent      0.864     0.915     0.889      2668

    accuracy                          0.793      5162
   macro avg      0.753     0.740     0.746      5162
weighted avg      0.788     0.793     0.790      5162



In [ ]:
print_section("RANDOM FOREST (define + train)")

rf_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000)),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

rf_model.fit(X_train_text, y_train)

print("Random Forest model trained on", len(X_train_text), "emails.")




RANDOM FOREST (define + train)


Random Forest model trained on 20644 emails.


In [ ]:
print_section("RANDOM FOREST (evaluation)")

y_train_pred_rf = rf_model.predict(X_train_text)
y_test_pred_rf  = rf_model.predict(X_test_text)

rf_train_acc = accuracy_score(y_train, y_train_pred_rf)
rf_test_acc  = accuracy_score(y_test, y_test_pred_rf)
rf_train_f1  = f1_score(y_train, y_train_pred_rf, average="macro")
rf_test_f1   = f1_score(y_test, y_test_pred_rf, average="macro")

print("Random Forest – Train Acc:", f"{rf_train_acc:.3f}", "Test Acc:", f"{rf_test_acc:.3f}")
print("Random Forest – Train F1 :", f"{rf_train_f1:.3f}",  "Test F1 :", f"{rf_test_f1:.3f}")

print("\nRandom Forest – Classification report (test set):\n")
print(classification_report(y_test, y_test_pred_rf, digits=3, zero_division=0))




RANDOM FOREST (evaluation)


Random Forest – Train Acc: 0.980 Test Acc: 0.769
Random Forest – Train F1 : 0.973 Test F1 : 0.702

Random Forest – Classification report (test set):

              precision    recall  f1-score   support

     deleted      0.718     0.453     0.556       988
       inbox      0.727     0.642     0.682      1506
        sent      0.796     0.957     0.869      2668

    accuracy                          0.769      5162
   macro avg      0.747     0.684     0.702      5162
weighted avg      0.761     0.769     0.754      5162



In [ ]:
print_section("DISTILBERT SETUP (labels + datasets)")

#Bidirectional Encoder Representations from Transformers
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

# 1. Encode text labels (sent/inbox/deleted) → integers 0,1,2
label_encoder = LabelEncoder()
y_train_ids = label_encoder.fit_transform(y_train)
y_test_ids  = label_encoder.transform(y_test)

print("Label classes:", list(label_encoder.classes_))
print("Encoded as   :", list(range(len(label_encoder.classes_))))

# 2. Load tokenizer
bert_model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

MAX_LEN = 256      # max tokens per email
BERT_TRAIN_MAX = 10000   # limit train size for speed (you can increase if GPU is strong)

# 3. Subsample training data for BERT to save time
if len(X_train_text) > BERT_TRAIN_MAX:
    train_sample_idx = X_train_text.sample(n=BERT_TRAIN_MAX, random_state=RANDOM_STATE).index
    X_train_bert = X_train_text.loc[train_sample_idx]
    y_train_bert = y_train_ids[[list(X_train_text.index).index(i) for i in train_sample_idx]]
    print(f"Using a subset of {len(X_train_bert)} training emails for DistilBERT.")
else:
    X_train_bert = X_train_text
    y_train_bert = y_train_ids
    print(f"Using full training set of {len(X_train_bert)} emails for DistilBERT.")

# Test set: always keep full for fair comparison
X_test_bert = X_test_text
y_test_bert = y_test_ids

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = int(self.labels[idx])

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}  # remove batch dim
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

# 4. Create train and test datasets for BERT
train_dataset_bert = EmailDataset(X_train_bert, y_train_bert, tokenizer, max_len=MAX_LEN)
test_dataset_bert  = EmailDataset(X_test_bert,  y_test_bert,  tokenizer, max_len=MAX_LEN)

print("BERT train dataset size:", len(train_dataset_bert))
print("BERT test dataset size :", len(test_dataset_bert))




DISTILBERT SETUP (labels + datasets)


Label classes: ['deleted', 'inbox', 'sent']
Encoded as   : [0, 1, 2]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Using a subset of 10000 training emails for DistilBERT.
BERT train dataset size: 10000
BERT test dataset size : 5162


In [ ]:
print_section("DISTILBERT ")

from transformers import TrainingArguments

num_labels = len(label_encoder.classes_)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=num_labels
)

# Simpler TrainingArguments compatible with older transformers versions
training_args = TrainingArguments(
    output_dir="./distilbert_email_clf",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs"

)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset_bert,
    eval_dataset=test_dataset_bert,
    tokenizer=tokenizer,
)

print("Starting DistilBERT training...")
train_result = trainer.train()
print("DistilBERT training finished.")




DISTILBERT 




model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1706012229.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting DistilBERT training...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

In [ ]:
print_section("DISTILBERT (evaluation)")

import numpy as np

# 1. Get predictions from Trainer
train_pred = trainer.predict(train_dataset_bert)
test_pred  = trainer.predict(test_dataset_bert)

# logits → class ids
y_pred_train_ids = np.argmax(train_pred.predictions, axis=1)
y_pred_test_ids  = np.argmax(test_pred.predictions,  axis=1)

# 2. Compute metrics in label-id space
bert_train_acc = accuracy_score(y_train_bert, y_pred_train_ids)
bert_test_acc  = accuracy_score(y_test_bert,  y_pred_test_ids)
bert_train_f1  = f1_score(y_train_bert, y_pred_train_ids, average="macro")
bert_test_f1   = f1_score(y_test_bert,  y_pred_test_ids,  average="macro")

print("DistilBERT – Train Acc:", f"{bert_train_acc:.3f}", "Test Acc:", f"{bert_test_acc:.3f}")
print("DistilBERT – Train F1 :", f"{bert_train_f1:.3f}",  "Test F1 :", f"{bert_test_f1:.3f}")

# 3. Nice classification report using original text labels
y_test_true_labels = label_encoder.inverse_transform(y_test_bert)
y_test_pred_labels = label_encoder.inverse_transform(y_pred_test_ids)

print("\nDistilBERT – Classification report (test set):\n")
print(classification_report(y_test_true_labels, y_test_pred_labels,
                            digits=3, zero_division=0))


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print_section("CONFUSION MATRIX – DISTILBERT")

# 1. Convert integer IDs back to original text labels
y_test_true_labels = label_encoder.inverse_transform(y_test_bert)
y_test_pred_labels = label_encoder.inverse_transform(y_pred_test_ids)

# Use a fixed order for the axes
labels = ["sent", "inbox", "deleted"]

# 2. Compute confusion matrix
cm_bert = confusion_matrix(y_test_true_labels, y_test_pred_labels, labels=labels)
print("Confusion matrix (rows = true, cols = predicted):")
print(cm_bert)

# 3. Plot nicely
disp = ConfusionMatrixDisplay(confusion_matrix=cm_bert, display_labels=labels)
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("DistilBERT – Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
print_section("OVERALL MODEL COMPARISON (NB, SVM, RF, DistilBERT)")

results_all = pd.DataFrame({
    "Model": ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"],
    "Train_Acc": [nb_train_acc, svm_train_acc, rf_train_acc, bert_train_acc],
    "Test_Acc":  [nb_test_acc,  svm_test_acc,  rf_test_acc,  bert_test_acc],
    "Train_F1":  [nb_train_f1,  svm_train_f1,  rf_train_f1,  bert_train_f1],
    "Test_F1":   [nb_test_f1,   svm_test_f1,   rf_test_f1,   bert_test_f1],
}).set_index("Model")

results_all


In [ ]:
print_section("VISUAL MODEL COMPARISON")

# Ensure models are in a sensible order
results_all_plot = results_all.copy().loc[
    ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"]
]

# Bar chart for Test F1
plt.figure(figsize=(6,4))
results_all_plot["Test_F1"].plot(kind="bar")
plt.title("Model comparison – Test Macro F1")
plt.ylabel("Macro F1 score")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# 2Bar chart for Test Accuracy
plt.figure(figsize=(6,4))
results_all_plot["Test_Acc"].plot(kind="bar")
plt.title("Model comparison – Test Accuracy")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
print_section("METRICS (Acc, Precision, Recall, F1) ")

from sklearn.metrics import precision_recall_fscore_support

def macro_metrics(y_true, y_pred):
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)
    return acc, prec, rec, f1

# Naive Bayes
nb_acc, nb_prec, nb_rec, nb_f1 = macro_metrics(y_test, y_test_pred_nb)

# Linear SVM
svm_acc, svm_prec, svm_rec, svm_f1 = macro_metrics(y_test, y_test_pred_svm)

# Random Forest
rf_acc, rf_prec, rf_rec, rf_f1 = macro_metrics(y_test, y_test_pred_rf)

# DistilBERT (we already inverse-transformed predictions)
bert_acc, bert_prec, bert_rec, bert_f1 = macro_metrics(y_test_true_labels, y_test_pred_labels)

metrics_radar = pd.DataFrame({
    "Model": ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"],
    "Accuracy":  [nb_acc,   svm_acc,   rf_acc,   bert_acc],
    "Precision": [nb_prec,  svm_prec,  rf_prec,  bert_prec],
    "Recall":    [nb_rec,   svm_rec,   rf_rec,   bert_rec],
    "F1":        [nb_f1,    svm_f1,    rf_f1,    bert_f1],
}).set_index("Model")

metrics_radar


In [ ]:
print_section("RADAR PLOT (Acc, Precision, Recall, F1)")

labels = ["Accuracy", "Precision", "Recall", "F1"]
num_vars = len(labels)

# Angles for the axes (one per metric)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # close the loop

def add_model_radar(values, label, linestyle="-"):
    # values: list of 4 metric values between 0 and 1
    vals = values + values[:1]  # close the loop
    plt.plot(angles, vals, linewidth=2, linestyle=linestyle, label=label)
    plt.fill(angles, vals, alpha=0.1)

plt.figure(figsize=(7, 7))
ax = plt.subplot(111, polar=True)

# Draw one axis per variable + add labels
plt.xticks(angles[:-1], labels)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], color="grey", size=8)
plt.ylim(0, 1)

# Add each model
for model_name in metrics_radar.index:
    vals = metrics_radar.loc[model_name, labels].tolist()
    add_model_radar(vals, model_name)

plt.title("Model comparison – Accuracy, Precision, Recall, F1 (macro)", y=1.08)
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


In [ ]:
print_section("GROUPED BAR: Accuracy, Precision, Recall, F1")

metrics_plot = metrics_radar.copy().loc[
    ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"]
]

models = metrics_plot.index.tolist()
metrics = ["Accuracy", "Precision", "Recall", "F1"]

x = np.arange(len(models))
width = 0.2

plt.figure(figsize=(8, 5))

for i, m in enumerate(metrics):
    plt.bar(x + i*width - width*1.5, metrics_plot[m], width, label=m)

plt.xticks(x, models, rotation=45, ha="right")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model comparison – macro Accuracy / Precision / Recall / F1")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
print_section("GROUPED BAR: Train vs Test F1 per model")

results_plot = results_all.copy().loc[
    ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"]
]

models = results_plot.index.tolist()
x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(7, 5))

plt.bar(x - width/2, results_plot["Train_F1"], width, label="Train F1")
plt.bar(x + width/2, results_plot["Test_F1"],  width, label="Test F1")

plt.xticks(x, models, rotation=45, ha="right")
plt.ylim(0, 1)
plt.ylabel("Macro F1 score")
plt.title("Train vs Test macro F1 – generalisation behaviour")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
print_section("METADATA FEATURE ENGINEERING)")

from pandas.api.types import is_datetime64_any_dtype

# Work on the classification dataset (sent / inbox / deleted)
meta_df = df_clf.copy()

# Parse the date column safely (timezone-aware is fine)
meta_df["date_parsed"] = pd.to_datetime(meta_df["date"], errors="coerce", utc=True)
print("date_parsed dtype:", meta_df["date_parsed"].dtype)

# Check using pandas helper
if is_datetime64_any_dtype(meta_df["date_parsed"]):
    # We have real datetimes → we can use .dt
    meta_df["year"]      = meta_df["date_parsed"].dt.year.fillna(0).astype(int)
    meta_df["month"]     = meta_df["date_parsed"].dt.month.fillna(0).astype(int)
    meta_df["dayofweek"] = meta_df["date_parsed"].dt.dayofweek.fillna(0).astype(int)   # 0=Mon
    meta_df["hour"]      = meta_df["date_parsed"].dt.hour.fillna(0).astype(int)
else:
    # Fallback: if something is still wrong, just set zeros
    print("Date parsing did not produce datetime64; using 0 for date-based features.")
    meta_df["year"]      = 0
    meta_df["month"]     = 0
    meta_df["dayofweek"] = 0
    meta_df["hour"]      = 0

# Internal vs external sender/receiver
def is_internal(addr):
    addr = str(addr).lower()
    return int("@enron.com" in addr)

meta_df["from_internal"] = meta_df["from"].apply(is_internal)
meta_df["to_internal"]   = meta_df["to"].apply(is_internal)

# Length features (already present from df_clean → df_clf)
print("Length columns present:", [c for c in meta_df.columns if "len" in c])

# Select metadata feature columns
meta_features = [
    "year", "month", "dayofweek", "hour",
    "subject_len", "body_len",
    "from_internal", "to_internal"
]

X_meta_full = meta_df[meta_features].fillna(0)

print("Metadata feature matrix shape:", X_meta_full.shape)
display(X_meta_full.head())

# Use the SAME indices as the text split
X_train_meta = X_meta_full.loc[train_idx]
X_test_meta  = X_meta_full.loc[test_idx]

print("X_train_meta shape:", X_train_meta.shape)
print("X_test_meta shape :", X_test_meta.shape)


In [ ]:
print_section(" METADATA-ONLY MODEL (Random Forest)")

meta_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

meta_model.fit(X_train_meta, y_train)

y_train_pred_meta = meta_model.predict(X_train_meta)
y_test_pred_meta  = meta_model.predict(X_test_meta)

meta_train_acc = accuracy_score(y_train, y_train_pred_meta)
meta_test_acc  = accuracy_score(y_test,  y_test_pred_meta)
meta_train_f1  = f1_score(y_train, y_train_pred_meta, average="macro")
meta_test_f1   = f1_score(y_test,  y_test_pred_meta,  average="macro")

print("Metadata-only RF – Train Acc:", f"{meta_train_acc:.3f}", "Test Acc:", f"{meta_test_acc:.3f}")
print("Metadata-only RF – Train F1 :", f"{meta_train_f1:.3f}",  "Test F1 :", f"{meta_test_f1:.3f}")

print("\nMetadata-only RF – Classification report (test set):\n")
print(classification_report(y_test, y_test_pred_meta, digits=3, zero_division=0))


In [ ]:
print_section("PREPARE TEXT + METADATA DATAFRAME")

X_all = meta_df[["text"]].copy()
for col in meta_features:
    X_all[col] = X_meta_full[col]

X_train_all = X_all.loc[train_idx]
X_test_all  = X_all.loc[test_idx]

print("X_train_all shape:", X_train_all.shape)
print("Columns:", X_train_all.columns.tolist())


In [ ]:
from sklearn.compose import ColumnTransformer

print_section("TEXT + METADATA SVM")

text_col = "text"
meta_cols = meta_features

preprocess_all = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), text_col),
        ("meta", "passthrough", meta_cols),
    ]
)

svm_text_meta_model = Pipeline([
    ("preprocess", preprocess_all),
    ("clf", LinearSVC())
])

svm_text_meta_model.fit(X_train_all, y_train)

y_train_pred_svm_tm = svm_text_meta_model.predict(X_train_all)
y_test_pred_svm_tm  = svm_text_meta_model.predict(X_test_all)

svm_tm_train_acc = accuracy_score(y_train, y_train_pred_svm_tm)
svm_tm_test_acc  = accuracy_score(y_test,  y_test_pred_svm_tm)
svm_tm_train_f1  = f1_score(y_train, y_train_pred_svm_tm, average="macro")
svm_tm_test_f1   = f1_score(y_test,  y_test_pred_svm_tm,  average="macro")

print("Text+Meta SVM – Train Acc:", f"{svm_tm_train_acc:.3f}", "Test Acc:", f"{svm_tm_test_acc:.3f}")
print("Text+Meta SVM – Train F1 :", f"{svm_tm_train_f1:.3f}",  "Test F1 :", f"{svm_tm_test_f1:.3f}")

print("\nText+Meta SVM – Classification report (test set):\n")
print(classification_report(y_test, y_test_pred_svm_tm, digits=3, zero_division=0))


In [ ]:
print_section("TEXT vs METADATA EXPERIMENT SUMMARY")

feature_exp = pd.DataFrame({
    "Model": [
        "Metadata-only RF",
        "Text-only SVM (TF-IDF)",
        "Text+Metadata SVM",
        "Text-only DistilBERT"
    ],
    "Test_Acc": [
        meta_test_acc,
        svm_test_acc,
        svm_tm_test_acc,
        bert_test_acc
    ],
    "Test_F1": [
        meta_test_f1,
        svm_test_f1,
        svm_tm_test_f1,
        bert_test_f1
    ]
}).set_index("Model")

feature_exp


In [ ]:
print_section("EXPERIMENT: TEXT vs TEXT+METADATA (SVM)")

exp2_table = pd.DataFrame({
    "Experiment": [
        "E1: Text only (subject+body)",
        "E2: Text + metadata (all cols)"
    ],
    "Test_Acc": [svm_test_acc,  svm_tm_test_acc],
    "Test_F1":  [svm_test_f1,   svm_tm_test_f1],
}).set_index("Experiment")

exp2_table


In [ ]:
plt.figure(figsize=(5,4))
exp2_table["Test_F1"].plot(kind="bar")
plt.ylim(0,1)
plt.ylabel("Macro F1")
plt.title("Experiment 1 vs 2 – SVM Test F1")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -------------------------
# 1) Load df_small
# -------------------------
if "parsed" in globals() and isinstance(parsed, pd.DataFrame):
    df_small = parsed.copy()
else:
    dfs = [(n, obj) for n, obj in globals().items() if isinstance(obj, pd.DataFrame)]
    if not dfs:
        raise NameError("No pandas DataFrames found. Load your dataset first (e.g., parsed).")
    name, df_small = max(dfs, key=lambda x: x[1].shape[0])
    df_small = df_small.copy()
    print(f"Using df: {name}")

print("✅ df_small shape:", df_small.shape)

# -------------------------
# 2) Create body_clean & subject_clean
# -------------------------
def clean_text(s: pd.Series) -> pd.Series:
    s = s.astype(str).fillna("")
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.replace(r"http\S+", " ", regex=True)
    s = s.str.replace(r"[^\w\s@.$%+-]", " ", regex=True)
    s = s.str.lower().str.strip()
    return s

if "body" not in df_small.columns:
    raise KeyError(f"No 'body' column found. Columns: {list(df_small.columns)}")

if "body_clean" not in df_small.columns:
    df_small["body_clean"] = clean_text(df_small["body"])
    print("✅ Created body_clean")

if "subject" in df_small.columns and "subject_clean" not in df_small.columns:
    df_small["subject_clean"] = clean_text(df_small["subject"])
    print("✅ Created subject_clean")

# -------------------------
# 3) Build spam_score (proxy)
# -------------------------
SPAM_PATTERNS = [
    r"\bunsubscribe\b", r"\bclick here\b", r"\bfree\b", r"\bwin\b", r"\bwinner\b",
    r"\bprize\b", r"\bclaim\b", r"\blimited time\b", r"\boffer\b", r"\bdeal\b",
    r"\bdiscount\b", r"\bpromo\b", r"\bpromotion\b", r"\bguarantee\b",
    r"\bcredit\b", r"\bloan\b", r"\bmortgage\b", r"\binvest\b", r"\bprofit\b",
    r"\bearn\b", r"\bcasino\b", r"\bgambling\b", r"\bbonus\b",
    r"\bact now\b", r"\burgent\b", r"\bcongratulations\b", r"\bselected\b",
]
spam_re = re.compile("|".join(SPAM_PATTERNS), flags=re.IGNORECASE)
url_re = re.compile(r"(http[s]?://|www\.)", flags=re.IGNORECASE)

def spam_score_row(subj: str, body: str) -> int:
    text = f"{subj} {body}".strip()
    score = 0
    if spam_re.search(text):
        score += 2
    if url_re.search(text):
        score += 1
    if "unsubscribe" in text:
        score += 2
    if any(k in text for k in ["limited time", "special offer", "buy now", "sale", "discount"]):
        score += 1
    if text.count("!") >= 3:
        score += 1
    return score

subj_series = df_small["subject_clean"] if "subject_clean" in df_small.columns else pd.Series([""] * len(df_small))
df_small["spam_score"] = [spam_score_row(s, b) for s, b in zip(subj_series.astype(str), df_small["body_clean"].astype(str))]

# -------------------------
# 4) Create HARDER proxy labels (more overlap = lower accuracy)
#   - spam if score >= 3
#   - not_spam if score <= 1
#   - drop score == 2 (uncertain)
# -------------------------
df_small["spam_label_hard"] = pd.Series(pd.NA, index=df_small.index, dtype="object")
df_small.loc[df_small["spam_score"] >= 3, "spam_label_hard"] = "spam"
df_small.loc[df_small["spam_score"] <= 1, "spam_label_hard"] = "not_spam"
df_hard = df_small.dropna(subset=["spam_label_hard"]).copy()

df_hard["y"] = df_hard["spam_label_hard"].map({"spam": 1, "not_spam": 0}).astype(int)
print(f"✅ hard-labeled rows: {len(df_hard)} | spam rate: {df_hard['y'].mean():.3f}")

if df_hard["y"].nunique() < 2:
    raise ValueError("Need both classes after hard labeling. Adjust thresholds.")

# -------------------------
# 5) Make BALANCED dataset for meaningful accuracy (50/50)
# -------------------------
n = min(df_hard["y"].value_counts().min(), 8000)  # cap to keep it fast
df_bal = pd.concat([
    df_hard[df_hard["y"] == 0].sample(n, random_state=42),
    df_hard[df_hard["y"] == 1].sample(n, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ balanced dataset: {df_bal.shape} | spam rate: {df_bal['y'].mean():.3f}")

# -------------------------
# 6) Split (group by sender if available to reduce leakage)
# -------------------------
FEATURE_COLS = ["body_clean"]
if "subject_clean" in df_bal.columns:
    FEATURE_COLS.append("subject_clean")

group_col = "from" if "from" in df_bal.columns else None

if group_col:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
    train_idx, test_idx = next(gss.split(df_bal, df_bal["y"], groups=df_bal[group_col].astype(str)))
    train_df = df_bal.iloc[train_idx].copy()
    test_df  = df_bal.iloc[test_idx].copy()
else:
    train_df, test_df = train_test_split(df_bal, test_size=0.25, random_state=42, stratify=df_bal["y"])

# further split train -> train/val for threshold tuning
train_df, val_df = train_test_split(train_df, test_size=0.25, random_state=42, stratify=train_df["y"])

for c in FEATURE_COLS:
    train_df[c] = train_df[c].fillna("")
    val_df[c]   = val_df[c].fillna("")
    test_df[c]  = test_df[c].fillna("")
# -------------------------
# 7) Train a slightly stronger (still controlled) model
# -------------------------
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

transformers = [
    # body: keep mostly simple but allow more features
    ("body", TfidfVectorizer(ngram_range=(1,1), min_df=2, max_features=20000), "body_clean"),
]

# subject: allow bigrams (helps phrases like "sale alert", "special offer")
if "subject_clean" in FEATURE_COLS:
    transformers.append(
        ("subj", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=8000), "subject_clean")
    )

preprocess = ColumnTransformer(transformers=transformers, remainder="drop")

spam_clf = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight=None))
])

spam_clf.fit(train_df[FEATURE_COLS], train_df["y"])

# -------------------------
# 8) Tune threshold to maximize VAL accuracy (balanced)
# -------------------------
p_val = spam_clf.predict_proba(val_df[FEATURE_COLS])[:, 1]
y_val = val_df["y"].values

thresholds = np.linspace(0.05, 0.95, 361)
accs = np.array([accuracy_score(y_val, (p_val >= t).astype(int)) for t in thresholds])

t_best = float(thresholds[accs.argmax()])
acc_best = float(accs.max())
print(f"✅ Best threshold: {t_best:.3f} | VAL accuracy: {acc_best:.4f}")

# -------------------------
# 9) Final test report
# -------------------------
p_test = spam_clf.predict_proba(test_df[FEATURE_COLS])[:, 1]
y_test = test_df["y"].values
y_pred = (p_test >= t_best).astype(int)

print("\n=== BALANCED TEST RESULTS (updated) ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))


In [ ]:
# Confusion Matrix (counts) + heatmap
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# If you already have y_test and y_pred from your model:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (rows=true, cols=pred):\n", cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not_spam", "spam"])
disp.plot(values_format="d")
plt.title("Spam vs Not-Spam Confusion Matrix")
plt.show()


In [ ]:
print_section("BUILD SUMMARISATION DATASET")

import re
import torch
from transformers import pipeline

#Start from master cleaned data
sum_df = df_clean.copy()

# Focus on folders where summarisation makes sense (e.g. inbox + sent)
sum_df = sum_df[sum_df["folder_label"].isin(["inbox", "sent"])].copy()

# Keep only reasonably long emails so that summarisation is meaningful
LONG_BODY_THRESHOLD = 400   # characters (tweak if needed)
sum_df = sum_df[sum_df["body_len"] >= LONG_BODY_THRESHOLD].copy()

print("Summarisation candidate set shape:", sum_df.shape)
display(sum_df[["folder_label", "subject", "body_len"]].head())

# For experiments: sample a manageable subset
N_SUM_EXAMPLES = 30
sum_examples = sum_df.sample(
    n=min(N_SUM_EXAMPLES, len(sum_df)),
    random_state=RANDOM_STATE
).copy()

print("\nSample of emails chosen for summarisation:")
display(sum_examples[["folder_label", "subject", "body_len"]].head())

In [ ]:

print_section("PRE-CLEANING + BASELINE SUMMARISER")

def clean_for_summarisation(text: str) -> str:
    """
    Light cleaning of email bodies before summarisation.
    - remove common header / forwarding / routing lines
    - strip extra whitespace
    """
    if not isinstance(text, str):
        text = str(text)

    lines = text.splitlines()
    cleaned_lines = []
    header_patterns = (
        "----- Forwarded by",
        "---------------------- Forwarded by",
        "----- Original Message",
        "---- Original Message",
        "From:",
        "To:",
        "Cc:",
        "CC:",
        "Subject:",
        "Sent:",
        "Received:",
    )

    for line in lines:
        stripped = line.strip()
        if any(stripped.startswith(p) for p in header_patterns):
            continue
        cleaned_lines.append(stripped)

    cleaned = " ".join(l for l in cleaned_lines if l)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def baseline_summary(text: str, max_sentences: int = 3) -> str:
    """
    Very simple extractive baseline:
    - clean the body
    - split into sentences
    - return the first N sentences
    """
    if not isinstance(text, str):
        text = str(text)

    text = clean_for_summarisation(text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return ""

    # Rough sentence split on ., ?, !
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences:
        return ""

    return " ".join(sentences[:max_sentences])


# Quick sanity check on a few emails
N_SHOW_BASELINE = 3
for idx, row in sum_examples.head(N_SHOW_BASELINE).iterrows():
    print("\n" + "="*80)
    print(f"Email ID: {idx}")
    print(f"Folder : {row['folder_label']}")
    print(f"Subject: {row['subject']}")
    print(f"Body length: {row['body_len']}")
    print("-"*80)
    print("BASELINE SUMMARY (first 3 sentences):\n")
    print(baseline_summary(row["body"], max_sentences=3))
    print("\n" + "="*80)

In [ ]:

print_section("LOAD TRANSFORMER SUMMARISER PIPELINE")

# Smaller BART model fine-tuned for summarisation
SUM_MODEL_NAME = "sshleifer/distilbart-cnn-12-6"

summariser = pipeline(
    task="summarization",
    model=SUM_MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)

print("Loaded summarisation model:", SUM_MODEL_NAME)


print_section("TRANSFORMER SUMMARY FUNCTION")

def transformer_summary(
    text: str,
    min_length: int = 40,
    max_length: int = 120
) -> str:
    """
    Use DistilBART summarisation model to generate an abstractive summary.
    - clean the body
    - feed to the summariser (with truncation)
    """
    if not isinstance(text, str):
        text = str(text)

    text = clean_for_summarisation(text).strip()
    if not text:
        return ""

    result = summariser(
        text,
        min_length=min_length,
        max_length=max_length,
        do_sample=False,   # deterministic
        truncation=True,
    )
    return result[0]["summary_text"].strip()


In [ ]:


print_section("ORIGINAL vs BASELINE vs TRANSFORMER (EXAMPLES)")

N_SHOW_COMPARE = 3
MAX_ORIG_CHARS = 800

for idx, row in sum_examples.head(N_SHOW_COMPARE).iterrows():
    original_body = row["body"]

    base_sum = baseline_summary(original_body, max_sentences=3)
    bart_sum = transformer_summary(original_body, min_length=40, max_length=120)

    print("\n" + "="*120)
    print(f"Email ID : {idx}")
    print(f"Folder   : {row['folder_label']}")
    print(f"Subject  : {row['subject']}")
    print(f"Body len : {row['body_len']} characters")
    print("="*120)

    # Original text (truncated)
    cleaned_original = clean_for_summarisation(original_body)
    print("\nORIGINAL BODY (cleaned, first "
          f"{MAX_ORIG_CHARS} chars):\n")
    print(cleaned_original[:MAX_ORIG_CHARS] + ("..." if len(cleaned_original) > MAX_ORIG_CHARS else ""))

    # Baseline
    print("\n" + "-"*120)
    print("BASELINE SUMMARY (lead-3 sentences):\n")
    print(base_sum)

    # Transformer
    print("\n" + "-"*120)
    print("TRANSFORMER SUMMARY (DistilBART):\n")
    print(bart_sum)

    print("\n" + "="*120)



In [ ]:
print_section("SAVE ARTEFACTS LOCALLY")

import joblib
from transformers import DistilBertTokenizerFast

#save datasets
df_clean.to_csv("clean_emails.csv", index=False)
df_clf.to_csv("clf_dataset.csv", index=False)

#save classical SVM model (use your actual SVM variable name)
joblib.dump(svm_model, "svm_tfidf_model.joblib")   # or svm_clf etc.

#save DistilBERT classifier + tokenizer

# If tokenizer variable doesn't exist yet, recreate it
try:
    bert_tokenizer
except NameError:
    # use the same base model name you used when training
    bert_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# Now save both model and tokenizer into one folder
bert_model.save_pretrained("distilbert_email_clf")
bert_tokenizer.save_pretrained("distilbert_email_clf")

# save metrics + experiments
results_all.to_csv("classification_results.csv", index=False)
feature_exp.to_csv("feature_experiments.csv", index=False)
exp2_table.to_csv("text_vs_textmeta_experiment.csv", index=False)

# -------------------------
# save SPAM prediction results (CSV)

spam_cols = [c for c in [
    "message_id","date","from","to","subject","body_clean",
    "spam_score","spam_pred_label","spam_proba"
] if c in df_small.columns]

df_small[spam_cols].to_csv("spam_results.csv", index=False)
print("Saved: spam_results.csv")


# build and save summarisation examples
summary_rows = []
N_SUM_SAVE = 10   # how many examples to save

for idx, row in sum_examples.head(N_SUM_SAVE).iterrows():
    body_cleaned = clean_for_summarisation(row["body"])
    base_sum = baseline_summary(row["body"], max_sentences=3)
    bart_sum = transformer_summary(row["body"], min_length=40, max_length=120)

    summary_rows.append({
        "id": idx,
        "folder_label": row["folder_label"],
        "subject": row["subject"],
        "body_len": row["body_len"],
        "body_cleaned_first_800": body_cleaned[:800],
        "baseline_summary": base_sum,
        "transformer_summary": bart_sum,
    })

summaries_df = pd.DataFrame(summary_rows)
summaries_df.to_csv("summaries_examples.csv", index=False)



print("Saved: clean_emails.csv, clf_dataset.csv, svm_tfidf_model.joblib, distilbert_email_clf/,")
print("       classification_results.csv, feature_experiments.csv, text_vs_textmeta_experiment.csv,")
print("       summaries_examples.csv")


In [ ]:
print_section("CONNECT TO AZURE BLOB STORAGE")

!pip install azure-storage-blob -q

import os
from azure.storage.blob import BlobServiceClient

#  connection string here (or use an environment variable)
AZURE_CONN_STR = "xxxxxxxxxxxxxxxxxxxxx"
CONTAINER_NAME = "email-ml-artifacts"   # the container

# BlobServiceClient
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONN_STR)

# the container client
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
    print(f"Container '{CONTAINER_NAME}' created.")
except Exception as e:
    print(f"Container '{CONTAINER_NAME}' may already exist:", e)


In [ ]:

print_section("UPLOAD ARTEFACTS TO AZURE")

import glob

def upload_file_to_blob(local_path, blob_path):
    """Upload a single local file to the given blob path."""
    with open(local_path, "rb") as data:
        container_client.upload_blob(
            name=blob_path,
            data=data,
            overwrite=True
        )
    print(f"Uploaded {local_path}  ->  {blob_path}")


# upload datasets
upload_file_to_blob("clean_emails.csv", "data/clean_emails.csv")
upload_file_to_blob("clf_dataset.csv", "data/clf_dataset.csv")

# upload classical SVM model
upload_file_to_blob("svm_tfidf_model.joblib", "models/svm_tfidf_model.joblib")

import os, glob

# upload DistilBERT model folder (only files, skip subdirs like checkpoint-500)
for local_file in glob.glob("distilbert_email_clf/*"):
    if not os.path.isfile(local_file):
        # skip directories such as 'checkpoint-500'
        print("Skipping directory:", local_file)
        continue
    fname = os.path.basename(local_file)
    blob_path = f"models/distilbert_email_clf/{fname}"
    upload_file_to_blob(local_file, blob_path)


# upload metrics / experiment tables
upload_file_to_blob("classification_results.csv", "metrics/classification_results.csv")
upload_file_to_blob("feature_experiments.csv", "metrics/feature_experiments.csv")
upload_file_to_blob("text_vs_textmeta_experiment.csv", "metrics/text_vs_textmeta_experiment.csv")
# upload spam prediction results
upload_file_to_blob("spam_results.csv", "results/spam_results.csv")


# upload summarisation examples
upload_file_to_blob("summaries_examples.csv", "summaries/summaries_examples.csv")

print("\nAll artefacts uploaded to Azure Blob Storage.")


In [ ]:
import os

root = "artifacts"  # change if your folder name differs

for dirpath, dirnames, filenames in os.walk(root):
    level = dirpath.replace(root, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    subindent = "  " * (level + 1)
    for f in sorted(filenames)[:50]:  # avoids printing too much
        print(f"{subindent}{f}")


In [ ]:
# -------------------------------
# Figure 3: Missing values summary (subject, body, text)
# (bar chart + optional table output)
# -------------------------------
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1) Pick an existing dataframe from your notebook (no renaming needed)
df_for_nulls = None
for nm in ["df_clean", "df_small", "df_bal", "clf_dataset", "parsed", "df"]:
    if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
        df_for_nulls = globals()[nm]
        df_name = nm
        break

assert df_for_nulls is not None, "No dataframe found. Expected one of: df_clean, df_small, df_bal, clf_dataset, parsed, df"

# 2) Choose the key fields that exist in YOUR dataframe (keeps your variable names)
candidate_cols = [
    "subject", "subject_clean",
    "body", "body_clean",
    "text", "text_clean",
    "combined_text", "email_text", "full_text"
]
cols = [c for c in candidate_cols if c in df_for_nulls.columns]

assert len(cols) > 0, f"None of the expected columns found in {df_name}. Available columns: {list(df_for_nulls.columns)[:30]}..."

# 3) Missing values summary (count + %)
null_count = df_for_nulls[cols].isna().sum()
null_pct = (null_count / len(df_for_nulls) * 100).round(2)

null_summary = pd.DataFrame({
    "field": null_count.index,
    "missing_count": null_count.values,
    "missing_%": null_pct.values
}).sort_values("missing_count", ascending=False)

display(null_summary)  # screenshot this if you want it as a table

# 4) Bar chart
plt.figure(figsize=(7, 4))
plt.bar(null_summary["field"], null_summary["missing_count"])
plt.title(f"Figure 3 – Missing Values Summary ({df_name})")
plt.xlabel("Field")
plt.ylabel("Missing count")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()




In [ ]:
# ---------------------------------------------------------
# Figure 8.4: Model comparison table (Test Accuracy + Macro F1)
# Uses your existing notebook variables (results_all if available)
# ---------------------------------------------------------
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# 1) Build the comparison table
model_order = ["Naive Bayes", "Linear SVM", "Random Forest", "DistilBERT"]

if "results_all" in globals() and isinstance(results_all, pd.DataFrame):
    # Use the table you already created in the notebook
    comp = results_all.copy()
    # If "Model" is a column, set it as index (handles both formats)
    if "Model" in comp.columns:
        comp = comp.set_index("Model")
    comp = comp.loc[model_order, ["Test_Acc", "Test_F1"]].copy()
else:
    # Fallback: compute metrics directly from predictions if results_all is not in memory
    required = ["y_test", "y_test_pred_nb", "y_test_pred_svm", "y_test_pred_rf",
                "y_test_true_labels", "y_test_pred_labels"]
    missing = [v for v in required if v not in globals()]
    assert not missing, f"Missing variables for fallback metrics: {missing}"

    comp = pd.DataFrame({
        "Test_Acc": [
            accuracy_score(y_test, y_test_pred_nb),
            accuracy_score(y_test, y_test_pred_svm),
            accuracy_score(y_test, y_test_pred_rf),
            accuracy_score(y_test_true_labels, y_test_pred_labels),
        ],
        "Test_F1": [
            f1_score(y_test, y_test_pred_nb, average="macro"),
            f1_score(y_test, y_test_pred_svm, average="macro"),
            f1_score(y_test, y_test_pred_rf, average="macro"),
            f1_score(y_test_true_labels, y_test_pred_labels, average="macro"),
        ],
    }, index=model_order)

# 2) Clean formatting for report
figure_8_4 = comp.rename(columns={
    "Test_Acc": "Accuracy (Test)",
    "Test_F1": "Macro F1 (Test)"
}).round(3)

display(figure_8_4)  # Screenshot this table for Figure 8.4

# 3) Save as CSV + PNG table image (nice for report insertion)
OUT_METRICS = "artifacts/metrics"
OUT_PLOTS = "artifacts/plots"
os.makedirs(OUT_METRICS, exist_ok=True)
os.makedirs(OUT_PLOTS, exist_ok=True)

csv_path = f"{OUT_METRICS}/figure_8_4_model_comparison.csv"
figure_8_4.to_csv(csv_path, index=True)

# PNG table export
fig, ax = plt.subplots(figsize=(6.8, 2.0))
ax.axis("off")
tbl = ax.table(
    cellText=figure_8_4.values,
    rowLabels=figure_8_4.index.tolist(),
    colLabels=figure_8_4.columns.tolist(),
    cellLoc="center",
    loc="center"
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.4)
plt.title("Figure 8.4 – Model comparison (Test Accuracy and Macro F1)", pad=12)
png_path = f"{OUT_PLOTS}/figure_8_4_model_comparison.png"
plt.savefig(png_path, dpi=200, bbox_inches="tight")
plt.close()

print("✅ Saved:", csv_path)
print("✅ Saved:", png_path)


In [ ]:
# ---------------------------------------------------------
# Figure 8.6: Email length distribution (Histogram)
# Uses existing variables in your notebook (df_clean / df_small / clf_dataset / parsed / df_bal)
# Plots text_len if available, otherwise body_len; computes if missing.
# ---------------------------------------------------------
import os
import pandas as pd
import matplotlib.pyplot as plt

# 1) Pick an existing dataframe (no renaming needed)
df_len = None
for nm in ["clf_dataset", "df_clean", "df_small", "df_bal", "parsed", "df"]:
    if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
        df_len = globals()[nm].copy()
        df_name = nm
        break
assert df_len is not None, "No dataframe found. Expected one of: clf_dataset, df_clean, df_small, df_bal, parsed, df"

# 2) Choose/compute a length column
if "text_len" in df_len.columns:
    len_col = "text_len"
elif "body_len" in df_len.columns:
    len_col = "body_len"
else:
    # compute from an existing text field
    if "text" in df_len.columns:
        base_text_col = "text"
    elif "body_clean" in df_len.columns:
        base_text_col = "body_clean"
    elif "body" in df_len.columns:
        base_text_col = "body"
    else:
        raise KeyError(f"No suitable text column found to compute length in {df_name}. "
                       f"Available columns: {list(df_len.columns)[:30]}...")

    len_col = "text_len" if base_text_col == "text" else "body_len"
    df_len[len_col] = df_len[base_text_col].fillna("").astype(str).str.len()

#print(f"Using dataframe: {df_name} | length column: {len_col} | rows: {len(df_len)}")

# 3) Plot histogram
vals = df_len[len_col].dropna()
# Optional: clip extreme outliers to make histogram readable
clip_max = int(vals.quantile(0.99)) if len(vals) > 0 else None
vals_plot = vals.clip(upper=clip_max) if clip_max else vals

plt.figure(figsize=(7, 4))
plt.hist(vals_plot, bins=40)
plt.title(f"Email length distribution ({len_col}) [{df_name}]")
plt.xlabel(f"{len_col} (characters)" + (f" (clipped at p99={clip_max})" if clip_max else ""))
plt.ylabel("Number of emails")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()



In [ ]:
# ---------------------------------------------------------
# Figure 8.7: % emails truncated summary (for transformers)
# Creates a small table: count + % of emails whose tokenized length > max_len
# Uses your existing text column (text / body_clean / body) and (if available) your tokenizer.
# ---------------------------------------------------------
import os
import pandas as pd

MAX_LEN = 256   # set to your DistilBERT max_length (change if you used 128/512)

# 1) Pick an existing dataframe from your notebook (no renaming needed)
df_trunc = None
for nm in ["clf_dataset", "df_clean", "df_small", "df_bal", "parsed", "df"]:
    if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
        df_trunc = globals()[nm].copy()
        df_name = nm
        break
assert df_trunc is not None, "No dataframe found. Expected one of: clf_dataset, df_clean, df_small, df_bal, parsed, df"

# 2) Choose the modelling text column used for DistilBERT
text_col = None
for c in ["text", "text_clean", "combined_text", "email_text", "body_clean", "body"]:
    if c in df_trunc.columns:
        text_col = c
        break
assert text_col is not None, f"No suitable text column found in {df_name}."

texts = df_trunc[text_col].fillna("").astype(str).tolist()

# 3) Get tokenizer (uses your existing tokenizer if present; otherwise loads DistilBERT tokenizer)
if "tokenizer" in globals():
    tok = tokenizer
else:
    from transformers import DistilBertTokenizerFast
    tok = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

# 4) Compute token lengths (no truncation here)
# Fast + memory-safe: process in batches
batch_size = 1024
token_lens = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    enc = tok(batch, truncation=False, padding=False, add_special_tokens=True)
    token_lens.extend([len(x) for x in enc["input_ids"]])

df_trunc["token_len"] = token_lens

# 5) Truncation summary table
n_total = len(df_trunc)
n_trunc = int((df_trunc["token_len"] > MAX_LEN).sum())
pct_trunc = round(n_trunc / n_total * 100, 2)

figure_8_7 = pd.DataFrame([{
    "dataset": df_name,
    "text_field": text_col,
    "max_len": MAX_LEN,
    "total_emails": n_total,
    "emails_exceeding_max_len": n_trunc,
    "%_truncated": pct_trunc
}])

display(figure_8_7)  # screenshot this small table for Figure 8.7


In [ ]:
# Pick 1 example (change idx if you want a different one)
ex = summaries_df.iloc[0].copy()

# Make a clean, report-ready table
table_df = pd.DataFrame({
    "Field": [
        "Email subject",
        "Body (first 200 chars)",
        "Lead-N (baseline) summary",
        "Transformer (BART) summary"
    ],
    "Example (from your dataset)": [
        ex.get("subject", ""),
        str(ex.get("body_cleaned_first_800", ""))[:200] + "…",
        ex.get("baseline_summary", ""),
        ex.get("transformer_summary", "")
    ]
})

display(table_df)


In [ ]:
ex = summaries_df.iloc[1].copy()

md = f"""| Field | Example (from your dataset) |
|---|---|
| Email subject | {ex.get('subject','')} |
| Body (first 200 chars) | {str(ex.get('body_cleaned_first_800',''))[:200]}… |
| Lead-N (baseline) summary | {ex.get('baseline_summary','')} |
| Transformer (BART) summary | {ex.get('transformer_summary','')} |
"""
print(md)


In [ ]:
# ---------------------------------------------------------
# Figure 5 (Table): Missing values + field types (subject, body, text)
# Prints a table (no bar chart) and saves it as CSV for your artefacts.
# Uses your existing dataframe variables.
# ---------------------------------------------------------
import os
import pandas as pd

# 1) Pick an existing dataframe (no renaming needed)
df_tbl = None
for nm in ["df_clean", "clf_dataset", "df_small", "df_bal", "parsed", "df"]:
    if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
        df_tbl = globals()[nm]
        df_name = nm
        break
assert df_tbl is not None, "No dataframe found. Expected one of: df_clean, clf_dataset, df_small, df_bal, parsed, df"

# 2) Choose key fields that exist
candidate_cols = ["subject", "subject_clean", "body", "body_clean", "text", "text_clean", "combined_text"]
cols = [c for c in candidate_cols if c in df_tbl.columns]

# Prefer exactly subject/body/text if present
preferred = [c for c in ["subject", "body", "text"] if c in df_tbl.columns]
cols = preferred if len(preferred) == 3 else cols

assert len(cols) > 0, f"No suitable columns found in {df_name}. Available: {list(df_tbl.columns)[:30]}..."

# 3) Build the table: dtype + missing + empty (after strip)
null_count = df_tbl[cols].isna().sum()
null_pct = (null_count / len(df_tbl) * 100).round(2)

# empty strings check (handles NaN safely)
empty_count = {}
for c in cols:
    empty_count[c] = df_tbl[c].fillna("").astype(str).str.strip().eq("").sum()
empty_count = pd.Series(empty_count)
empty_pct = (empty_count / len(df_tbl) * 100).round(2)

dtype = df_tbl[cols].dtypes.astype(str)

figure_5_table = pd.DataFrame({
    "field": cols,
    "dtype": [dtype[c] for c in cols],
    "missing_count": [int(null_count[c]) for c in cols],
    "missing_%": [float(null_pct[c]) for c in cols],
    "empty_count": [int(empty_count[c]) for c in cols],
    "empty_%": [float(empty_pct[c]) for c in cols],
}).sort_values(["missing_count", "empty_count"], ascending=False)

print(f"✅ Dataframe used: {df_name} | rows: {len(df_tbl)}")
display(figure_5_table)  # Screenshot this table for your report instead of a bar chart

